### IAGOS Flight Search & Download for IRISCC

This notebook provides tools to search and download IAGOS flight data focused on **CO (Carbon Monoxide) from fire sources** using profile data (vertical measurements during ascent/descent).

If you are not familiar with Python or notebooks, have a look at README.md next to this file.

## IRISCC Focus

This notebook is specifically designed for IRISCC with:
- **Fixed parameters**: `originCO = "fire"` and `type = "profiles"`
- **Variable parameters**: `start`, `end`, `bbox`, `level`, `region`

## Prerequisites

### Create a IAGOS account

1) Registrate following the link below: https://iagos.aeris-data.fr/registration/
2) Once it has been validated by the IAGOS team, you will be informed by receiving an email
3) On the authentication page https://iagos.aeris-data.fr/profile/, click on "forgot password" and create your password

### Authentication

Everything happens **inside this notebook**: run the authentication cell below, type your AERIS
e-mail and password, and press the button. No terminal, and nothing to edit in the source files.

Your credentials are never written into the package. By default they are kept in memory for this
session only; if you tick **Remember on this machine**, they are encrypted with a master password
of your choice into `~/.iagos/credentials.json` in your own home directory, readable by you alone.
From then on the cell asks only for the master password.

### Access to L1 data

L1 requires the `IAGOS_DATA_PROVIDER` role on your AERIS account; L2 and L4 are open to any
authenticated user. Without the role the server answers `403 - Access to L1 data requires
IAGOS_DATA_PROVIDER role`, and the CO ranking falls back to L2 automatically, so the notebook
still works.

## Server Configuration

- **Production URL**: `https://api.sedoo.fr/iagos-backend/v2.0/downloads`
- **Authentication**: Required via AERIS SSO

## API Services

### `/search` Service
Search for flights without downloading. Returns flight count and total data size.

### `/download-flights` Service
Download data files as a ZIP archive.
- **Maximum limit**: 1 GB per request
- **Validation**: Always search first to check data size

**Folder management: to avoid accumulating files across repeated runs, this notebook empties its `data/` and `exports/` folders at the start of each run. Use the export cell at the end to keep anything you want before running it again.**

> **Clearing the outputs.** Use *Edit → Clear Outputs of All Cells* in two situations:
>
> * **before running the notebook again**, so that the interactive widgets are rebuilt from scratch.
>   Widget state belongs to the kernel that created it, so leftovers from a previous run can make
>   controls stop responding or disappear;
> * **before saving**, so that the file stays small and does not carry stale widget state. Plotly
>   embeds a large JavaScript bundle into saved outputs, and reopening a notebook that stores widget
>   state shows "Error displaying widget: model not found".
>
> If controls are still unresponsive after clearing, restart the kernel (*Kernel → Restart Kernel*)
> and, should that not be enough, reload the browser page: the widgets live partly in the page
> itself, and a reload is sometimes the only way to clear what is left there.

In [ ]:
# The kernel keeps a handle on the directory it was started in. If that directory is replaced
# while the kernel is running - the shared workspace being republished, for instance - every
# relative path fails with "Stale file handle" or FileNotFoundError. Recover instead of leaving
# the user with an error that has nothing to do with what they were doing.
import os

try:
    os.getcwd()
except OSError:
    _nb = os.environ.get("JPY_SESSION_NAME", "")
    _target = os.path.dirname(_nb) if _nb else os.path.expanduser("~")
    if not os.path.isdir(_target):
        _target = os.path.expanduser("~")
    os.chdir(_target)
    print(f"The working directory had become stale and was reset to {_target}.")
    print("If anything still looks wrong, restart the kernel (Kernel -> Restart Kernel).")

# Identity of this notebook. The file itself carries no version number - only one version is
# published at a time - so the version is printed here instead, which is also what tells you
# whether the kernel is running the file you think it is.
NOTEBOOK_NAME = "IAGOS flight search"
NOTEBOOK_VERSION = "2.1"
NOTEBOOK_DATE = "2026-09-11"

print(f"{NOTEBOOK_NAME} - version {NOTEBOOK_VERSION} ({NOTEBOOK_DATE})")

#Install the libraries needed in the notebook. If it has already been done, you will receive messages saying "Requirement already satisfied"
import os
import importlib.util, subprocess, sys as _sys

# Install only the packages that are actually missing
# Every third-party module the notebook imports, including the ones that used to be taken for
# granted from the conda environment. Each VRE user has their own container, so what is present
# for one person may be missing for another.
_REQUIRED = {"keycloak": "python-keycloak", "matplotlib": "matplotlib",
             "numpy": "numpy", "pandas": "pandas", "xarray": "xarray",
             "h5netcdf": "h5netcdf", "h5py": "h5py", "ipywidgets": "ipywidgets"}
_missing = [p for m, p in _REQUIRED.items() if importlib.util.find_spec(m) is None]
if _missing:
    print(f"Installing: {', '.join(_missing)}")
    # pip often reports conflicts with packages already present in the image. They are
    # warnings about other packages, not a failure of this install: the imports below are
    # what actually decides whether the notebook can run.
    print("(pip may report dependency conflicts with packages already in the image - those are "
          "warnings, not failures; the imports below are the real check.)")
    subprocess.run([_sys.executable, "-m", "pip", "install", "--quiet", *_missing],
                   check=False, cwd=os.path.expanduser("~"))
else:
    print("All required packages are already installed.")

# Add project source code to Python path
import sys
from pathlib import Path
# The notebook sits at the root of the package, next to src/, for consistency with the other
# tools of the VRE. The check keeps it working from a notebooks/ subfolder as well.
_src = Path.cwd() / "src"
if not _src.is_dir():
    _src = Path.cwd().parent / "src"
sys.path.insert(0, str(_src))

# Import necessary libraries for this notebook
import json
import zipfile
import math
from pathlib import Path

# Import IAGOS authentication functions
from fr.aeris.iagos.auth import build_auth_header

# Import visualization libraries
import io
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("All imports successful")

from fr.aeris.iagos.iriscc import search_flights, download_flights, rank_flights_by_co

print("IRISCC utility functions imported successfully")

# API Configuration

BASE_URL = "https://api.sedoo.fr/iagos-backend/v2.0/downloads"
print(f"Using server: {BASE_URL}")
print(f"Base URL: {BASE_URL}")

# Downloaded archives go in a "data" folder next to the notebook, as in the other tools of this
# VRE. An earlier version put them in ~/.iagos/downloads to keep them out of the shared
# workspace; that is no longer needed now that users work from their own copy in their home,
# and a dotted folder is invisible in the file browser, reachable only from a terminal.
DATA_DIR = Path("data")

# Emptied at the start of each run, so repeated searches do not fill up the home directory:
# the L1/L2/L4 archives are tens of megabytes each. Set to False to keep what earlier runs
# downloaded. The stored credentials are elsewhere (~/.iagos/credentials.json) and untouched.
CLEAN_DATA_FOLDER = True

import shutil

if CLEAN_DATA_FOLDER and DATA_DIR.exists():
    freed = sum(f.stat().st_size for f in DATA_DIR.rglob("*") if f.is_file())
    shutil.rmtree(DATA_DIR, ignore_errors=True)
    if DATA_DIR.exists() and any(DATA_DIR.iterdir()):
        print("Note: some files could not be removed (still open elsewhere); continuing anyway.")
    elif freed:
        print(f"Cleared {freed / 1e6:.1f} MB from the previous run.")

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Download directory: {DATA_DIR}")
print(f"Free space in your home: {shutil.disk_usage(Path.home()).free / 1e9:.1f} GB")

# Figures drawn during the session, so the export cell at the end can write them out even
# though the inline backend closes them once a cell has finished.
DRAWN_FIGURES = {}

# Archives produced by the export cell at the end go here. The folder is emptied together with
# the rest, so the zips of previous runs do not pile up next to the notebook.
EXPORT_DIR = Path("exports")
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


## Authentication

Sign in with your **AERIS account** — the same e-mail and password you use on
[iagos.aeris-data.fr](https://iagos.aeris-data.fr/profile/).

Your credentials are never written into the source code of the package. By default they are kept in
memory for this session only. If you tick **Remember on this machine**, they are encrypted with a
master password of your choice and stored in `~/.iagos/credentials.json` in your own home directory,
readable by you alone — never in the shared workspace. From then on this cell asks only for the
master password.

If you have previously edited `authentication.py` by hand, those values are no longer used and can
be removed from the file.

In [ ]:
# =============================================================================
# AUTHENTICATION WITH AERIS SSO
# =============================================================================
# Your credentials are never written into the package source. They are either kept in memory
# for this session only, or stored encrypted in your own home directory
# (~/.iagos/credentials.json, readable by you alone), never in the shared workspace.

import json
import os
import stat
import time
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, HTML
from keycloak import KeycloakOpenID

from fr.aeris.iagos.auth import build_auth_header
from fr.aeris.iagos.auth.encryption import jasypt_encrypt, jasypt_decrypt

CRED_PATH = Path.home() / ".iagos" / "credentials.json"

# Keep any session already established: re-running this cell must not silently drop the token,
# which would make the cells below fail with a confusing "authorization required" from the server.
token_data = globals().get("token_data")
access_token = globals().get("access_token")
auth_headers = globals().get("auth_headers")


def _aeris_login(email, password):
    """Exchange AERIS credentials for an access token."""
    kc = KeycloakOpenID(server_url="https://sso.aeris-data.fr/auth/",
                        client_id="aeris-public", realm_name="aeris", verify=True)
    return kc.token(email, password)


def _store_credentials(email, password, master):
    """Encrypt the credentials into the user's home directory, readable by the owner only."""
    CRED_PATH.parent.mkdir(parents=True, exist_ok=True)
    payload = {"email": jasypt_encrypt(email, master),
               "password": jasypt_encrypt(password, master)}
    CRED_PATH.write_text(json.dumps(payload))
    CRED_PATH.chmod(stat.S_IRUSR | stat.S_IWUSR)          # 0600
    try:
        CRED_PATH.parent.chmod(stat.S_IRWXU)              # 0700
    except OSError:
        pass


def _load_credentials(master):
    """Decrypt the stored credentials.

    A wrong master password does not always make the decryption fail: it can simply return
    rubbish, which would then be rejected by AERIS with a misleading message. The decrypted
    e-mail is checked so that the real cause is reported instead.
    """
    payload = json.loads(CRED_PATH.read_text())
    email = jasypt_decrypt(payload["email"], master)
    password = jasypt_decrypt(payload["password"], master)
    if "@" not in email or not email.isprintable():
        raise ValueError("wrong master password")
    return email, password


def _finish(td, out):
    """Publish the token to the rest of the notebook."""
    global token_data, access_token, auth_headers, token_expires_at
    token_data = td
    token_expires_at = time.time() + float(td.get("expires_in") or 0)
    access_token = td["access_token"]
    auth_headers = build_auth_header("token", access_token)
    with out:
        expires = td.get("expires_in")
        print("Authenticated with AERIS SSO."
              + (f" The token is valid for about {int(expires) // 60} minutes." if expires else ""))
        print("You can move on to the next section.")


out = widgets.Output()

if auth_headers is not None:
    again_btn = widgets.Button(description="Sign in again", icon="refresh")

    def _on_again(_):
        global token_data, access_token, auth_headers
        token_data = access_token = auth_headers = None
        with out:
            out.clear_output()
            print("Session cleared. Re-run this cell to sign in again.")

    again_btn.on_click(_on_again)
    display(widgets.VBox([
        widgets.HTML("<b>Already signed in</b> for this session - nothing to do, "
                     "move on to the next section."),
        again_btn, out]))

elif CRED_PATH.exists():
    # Stored credentials: only the master password is needed
    master_pw = widgets.Password(description="Master password:",
                                 style={"description_width": "initial"},
                                 layout=widgets.Layout(width="380px"))
    unlock_btn = widgets.Button(description="Unlock and sign in", button_style="success",
                                icon="unlock")
    forget_btn = widgets.Button(description="Use a different account", icon="user")

    def _on_unlock(_):
        with out:
            out.clear_output()
            print("Signing in ...")
        try:
            email, password = _load_credentials(master_pw.value)
        except Exception:
            with out:
                out.clear_output()
                print("Could not decrypt the stored credentials - wrong master password?")
                print('If you have forgotten it, choose "Use a different account" to start over.')
            return
        try:
            _finish(_aeris_login(email, password), out)
        except Exception as exc:  # noqa: BLE001 - the reason must be readable
            with out:
                out.clear_output()
                print(f"AERIS rejected the sign-in ({type(exc).__name__}).")
                print("The stored credentials may be out of date; use a different account to "
                      "enter them again.")

    def _on_forget(_):
        CRED_PATH.unlink(missing_ok=True)
        with out:
            out.clear_output()
            print("Stored credentials deleted. Re-run this cell to enter new ones.")

    unlock_btn.on_click(_on_unlock)
    forget_btn.on_click(_on_forget)
    display(widgets.VBox([
        widgets.HTML(f"<b>Stored credentials found</b> in <code>{CRED_PATH}</code>."),
        master_pw, widgets.HBox([unlock_btn, forget_btn]), out]))

else:
    email_tx = widgets.Text(description="AERIS e-mail:", style={"description_width": "initial"},
                            layout=widgets.Layout(width="420px"))
    pw_tx = widgets.Password(description="AERIS password:",
                             style={"description_width": "initial"},
                             layout=widgets.Layout(width="420px"))
    remember_cb = widgets.Checkbox(value=False, indent=False,
                                   description="Remember on this machine (encrypted, in my home "
                                               "directory)")
    master_pw = widgets.Password(description="Master password:",
                                 style={"description_width": "initial"},
                                 layout=widgets.Layout(width="380px"), disabled=True)
    login_btn = widgets.Button(description="Sign in", button_style="success", icon="sign-in")

    def _on_remember(change):
        master_pw.disabled = not change["new"]

    def _on_login(_):
        with out:
            out.clear_output()
            print("Signing in ...")
        email, password = email_tx.value.strip(), pw_tx.value
        if not email or not password:
            with out:
                out.clear_output()
                print("Enter your AERIS e-mail and password.")
            return
        if remember_cb.value and not master_pw.value:
            with out:
                out.clear_output()
                print("Choose a master password, or untick the box to sign in without storing "
                      "anything.")
            return
        try:
            td = _aeris_login(email, password)
        except Exception as exc:  # noqa: BLE001 - the reason must be readable
            with out:
                out.clear_output()
                print(f"AERIS rejected the sign-in ({type(exc).__name__}).")
                print("Check your e-mail and password at https://iagos.aeris-data.fr/profile/ - "
                      "note that this is your AERIS account, not the master password.")
            return

        if remember_cb.value:
            _store_credentials(email, password, master_pw.value)
        _finish(td, out)
        if remember_cb.value:
            with out:
                print(f"Credentials stored, encrypted, in {CRED_PATH} (readable by you only).")
                print("Next time this cell will only ask for the master password.")

    remember_cb.observe(_on_remember, names="value")
    login_btn.on_click(_on_login)
    display(widgets.VBox([email_tx, pw_tx, remember_cb, master_pw, login_btn, out]))


def require_token(margin=30):
    """Fail with a readable message when the AERIS token has expired.

    AERIS tokens last about five minutes. An expired one makes the server answer 401, which
    reads as a permissions problem - it is not, it just means the sign-in has to be repeated.
    """
    if auth_headers is None:
        raise RuntimeError(
            "Not signed in. Go back to the Authentication section, press the sign-in button, "
            "and wait for the confirmation message before running this cell."
        )
    left = token_expires_at - time.time() if token_expires_at else 0
    if left < margin:
        raise RuntimeError(
            f"The AERIS token expired {abs(left) / 60:.0f} minute(s) ago. Press the sign-in "
            f"button again and re-run this cell - the server would otherwise answer 401, which "
            f"looks like a permissions problem but is not."
        )


In [ ]:
print("token present:", bool(access_token), "| length:", len(access_token or ""))
print("headers:", {k: (v[:25] + "...") if k == "Authorization" else v
                   for k, v in auth_headers.items()})

## API Reference for IRISCC

### Parameter Format

| Parameter | Type   | Required | Description                               | Example                         |
|-----------|--------|----------|-------------------------------------------|---------------------------------|
| start     | string | Yes      | Start date in ISO 8601 format             | `"2010-01-01T00:00:00"`         |
| end       | string | Yes      | End date in ISO 8601 format               | `"2020-12-31T23:59:59"`         |
| bbox      | string | Yes      | Bounding box: minLon,minLat,maxLon,maxLat | `"-180,-90,180,90"`             |
| level     | string | Yes      | Data level                                | `"L1"`, `"L2"` or `"L4"`        |
| region    | list   | No       | Region filter for CO source               | `["BONA", "EURO"]`              |
| l4Type    | list   | No       | L4 file types (L4 only)                   | `["ECMWF", "CO_CONTRIBUTIONS"]` |

**Fixed parameters for IRISCC:**
- `type = "profiles"` (vertical measurements during ascent/descent)
- `originCO = "fire"` (CO from biomass burning)

### L4 File Types

| Type             | Description                                                    |
|------------------|----------------------------------------------------------------|
| ECMWF            | Meteorological fields from ECMWF (temperature, wind, humidity) |
| CO_CONTRIBUTIONS | CO source attribution by region                                |
| PV               | Potential Vorticity                                            |

### Available Regions (GFED)

| Code | Region                            |
|------|-----------------------------------|
| AUST | Australia                         |
| BOAS | Boreal Asia                       |
| BONA | Boreal North America              |
| CEAM | Central America                   |
| CEAS | Central Asia                      |
| EQAS | Equatorial Asia                   |
| EURO | Europe                            |
| MIDE | Middle East                       |
| NHAF | Northern Hemisphere Africa        |
| NHSA | Northern Hemisphere South America |
| SEAS | Southeast Asia                    |
| SHAF | Southern Hemisphere Africa        |
| SHSA | Southern Hemisphere South America |
| TENA | Temperate North America           |

#### GFED Regions Map

![GFED Regions](images/GFED-regions.png)

### Download Limits

- Maximum **1GB** per download request
- Use `split_large_dataset()` function for larger requests

## Utility Functions

The IRISCC utility functions are imported from the `fr.aeris.iagos.iriscc` module:
- `search_flights`: Search for flights matching given criteria
- `download_flights`: Download flight data as ZIP files (skips if already downloaded)
- `split_large_dataset`: Split large requests into smaller periods under 1GB
- `rank_flights_by_co`: Analyze and rank flights by CO concentration

## Search and Download

This section demonstrates how to:
1. **Define search parameters** once for all operations
2. **Search** to verify data availability and size
3. **Download L1, L2, and L4** data with the same parameters
4. **Analyze** CO data and rank flights by concentration
5. **Visualize** the most interesting flights

In [ ]:
# =============================================================================
# SEARCH PARAMETERS
# =============================================================================
# Set them with the controls below and press "Confirm parameters". The values feed the search,
# the downloads and the plots further down, so nothing has to be edited in the cells.

from datetime import date, datetime

# GFED regions used by the CO source attribution, as listed in the API reference above
GFED_REGIONS = {
    "AUST": "Australia",
    "BOAS": "Boreal Asia",
    "BONA": "Boreal North America",
    "CEAM": "Central America",
    "CEAS": "Central Asia",
    "EQAS": "Equatorial Asia",
    "EURO": "Europe",
    "MIDE": "Middle East",
    "NHAF": "Northern Hemisphere Africa",
    "NHSA": "Northern Hemisphere South America",
    "SEAS": "Southeast Asia",
    "SHAF": "Southern Hemisphere Africa",
    "SHSA": "Southern Hemisphere South America",
    "TENA": "Temperate North America",
}

# Ready-made bounding boxes, "minLon,minLat,maxLon,maxLat"
BBOX_PRESETS = {
    "Global": "-180,-90,180,90",
    "Europe": "-25,30,45,72",
    "North America": "-170,15,-50,75",
    "Asia": "45,0,150,75",
    "Africa": "-20,-35,55,38",
    "South America": "-85,-56,-32,13",
    "Custom (type it below)": None,
}

start_dp = widgets.DatePicker(value=date(2025, 7, 23), description="Start date:",
                              style={"description_width": "initial"},
                              layout=widgets.Layout(width="280px"))
end_dp = widgets.DatePicker(value=date(2025, 7, 29), description="End date:",
                            style={"description_width": "initial"},
                            layout=widgets.Layout(width="280px"))

bbox_dd = widgets.Dropdown(options=list(BBOX_PRESETS), value="Global", description="Area:",
                           style={"description_width": "initial"},
                           layout=widgets.Layout(width="340px"))
bbox_tx = widgets.Text(value=BBOX_PRESETS["Global"], description="bbox:", disabled=True,
                       placeholder="minLon,minLat,maxLon,maxLat",
                       style={"description_width": "initial"},
                       layout=widgets.Layout(width="420px"))

region_sel = widgets.SelectMultiple(
    options=[(f"{code} - {name}", code) for code, name in sorted(GFED_REGIONS.items())],
    value=(), rows=8, description="CO source regions:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="480px"))
region_note = widgets.HTML(
    "<i>Leave empty for no region filter (fire CO from anywhere). "
    "Ctrl-click or cmd-click to select several.</i>")

confirm_params_btn = widgets.Button(description="Confirm parameters", button_style="success",
                                    icon="check")
params_out = widgets.Output()

if "SEARCH_PARAMS" not in globals():
    SEARCH_PARAMS = None


def _on_bbox_choice(change):
    preset = BBOX_PRESETS[change["new"]]
    bbox_tx.disabled = preset is not None      # editable only for "Custom"
    if preset is not None:
        bbox_tx.value = preset


def _on_confirm_params(_):
    global SEARCH_PARAMS
    with params_out:
        params_out.clear_output()
        if start_dp.value is None or end_dp.value is None:
            print("Choose both a start and an end date.")
            return
        if end_dp.value < start_dp.value:
            print("The end date is before the start date.")
            return

        bbox = bbox_tx.value.strip()
        parts = bbox.split(",")
        if len(parts) != 4:
            print('The bounding box must be "minLon,minLat,maxLon,maxLat", four numbers.')
            return
        try:
            lon0, lat0, lon1, lat1 = (float(p) for p in parts)
        except ValueError:
            print(f"The bounding box contains something that is not a number: {bbox}")
            return
        if not (-180 <= lon0 < lon1 <= 180 and -90 <= lat0 < lat1 <= 90):
            print(f"The bounding box is out of range or inverted: {bbox}")
            return

        # The API wants whole seconds; the end date is taken as the whole day
        SEARCH_PARAMS = {
            "start": f"{start_dp.value:%Y-%m-%d}T00:00:00",
            "end": f"{end_dp.value:%Y-%m-%d}T23:59:59",
            "bbox": bbox,
            "region": list(region_sel.value) or None,
        }
        print("Search parameters confirmed:")
        print(json.dumps(SEARCH_PARAMS, indent=2))
        span = (end_dp.value - start_dp.value).days + 1
        print(f"\n({span} day(s), "
              f"{'no region filter' if not region_sel.value else 'regions: ' + ', '.join(region_sel.value)})")


bbox_dd.observe(_on_bbox_choice, names="value")
confirm_params_btn.on_click(_on_confirm_params)

display(widgets.VBox([
    widgets.HBox([start_dp, end_dp]),
    bbox_dd, bbox_tx,
    region_sel, region_note,
    confirm_params_btn, params_out,
]))

with params_out:
    if SEARCH_PARAMS:
        print("Parameters already confirmed:")
        print(json.dumps(SEARCH_PARAMS, indent=2))
    else:
        print("Not confirmed yet - press the button before running the cells below.")

In [ ]:
require_token()

if SEARCH_PARAMS is None:
    raise RuntimeError(
        "Search parameters not confirmed. Go back to the parameters cell, set the dates, the "
        "area and the regions, and press 'Confirm parameters'."
    )

res_l1 = search_flights(
    BASE_URL, auth_headers,
    start=SEARCH_PARAMS["start"], end=SEARCH_PARAMS["end"],
    bbox=SEARCH_PARAMS["bbox"], level="L1", region=SEARCH_PARAMS["region"])
print(res_l1)

In [ ]:
require_token()

if SEARCH_PARAMS is None:
    raise RuntimeError(
        "Search parameters not confirmed. Go back to the parameters cell, set the dates, the "
        "area and the regions, and press 'Confirm parameters'."
    )

res_l2 = search_flights(
    BASE_URL, auth_headers,
    start=SEARCH_PARAMS["start"], end=SEARCH_PARAMS["end"],
    bbox=SEARCH_PARAMS["bbox"], level="L2", region=SEARCH_PARAMS["region"])
print(res_l2)

In [ ]:
require_token()

if SEARCH_PARAMS is None:
    raise RuntimeError(
        "Search parameters not confirmed. Go back to the parameters cell, set the dates, the "
        "area and the regions, and press 'Confirm parameters'."
    )

# Search to verify data availability
# This helps confirm the number of flights and total size before downloading

search_result = search_flights(
    BASE_URL, auth_headers,
    start=SEARCH_PARAMS["start"],
    end=SEARCH_PARAMS["end"],
    bbox=SEARCH_PARAMS["bbox"],
    level="L1",  # Search L1 first (preferred for CO data)
    region=SEARCH_PARAMS["region"]
)

if search_result:
    print("\nQuery details:")
    print(json.dumps(search_result['query'], indent=2))

### Download L1, L2, and L4 data

We download all three data levels with the same search parameters:
- **L2**: Validated data (preferred for CO)
- **L1**: Raw measurements (fallback when L2 has no valid CO). Note that these data aren't fully calibrated.
- **L4**: Regional CO contributions (for visualization)

> **Download folder management.** The `data/` folder next to this notebook is **emptied at the
> start of each run**, so that repeated searches do not fill up your home directory — the L1/L2/L4
> archives are tens of megabytes each. Set `CLEAN_DATA_FOLDER = False` in the first cell to keep
> what earlier runs downloaded.
>
> The data of a run are therefore gone as soon as you run the notebook again: **use the
> [export cell](#export) at the end to keep anything you want**, before starting a new search.

In [ ]:
require_token()

if SEARCH_PARAMS is None:
    raise RuntimeError(
        "Search parameters not confirmed. Go back to the parameters cell, set the dates, the "
        "area and the regions, and press 'Confirm parameters'."
    )

# Download L2 data (preferred for CO)

print("=" * 60)
print("Downloading L2 data...")
print("=" * 60)

zip_file_l2 = download_flights(
    BASE_URL, auth_headers,
    start=SEARCH_PARAMS["start"],
    end=SEARCH_PARAMS["end"],
    bbox=SEARCH_PARAMS["bbox"],
    level="L2",
    region=SEARCH_PARAMS["region"],
    output_dir=str(DATA_DIR)
)

print(f"\nL2 ZIP file: {zip_file_l2}")

# Download L1 data (fallback when L2 has no valid CO)

print("\n" + "=" * 60)
print("Downloading L1 data...")
print("=" * 60)

zip_file_l1 = download_flights(
    BASE_URL, auth_headers,
    start=SEARCH_PARAMS["start"],
    end=SEARCH_PARAMS["end"],
    bbox=SEARCH_PARAMS["bbox"],
    level="L1",
    region=SEARCH_PARAMS["region"],
    output_dir=str(DATA_DIR)
)

print(f"\nL1 ZIP file: {zip_file_l1}")

In [ ]:
require_token()

if SEARCH_PARAMS is None:
    raise RuntimeError(
        "Search parameters not confirmed. Go back to the parameters cell, set the dates, the "
        "area and the regions, and press 'Confirm parameters'."
    )

# Download L4 data (regional CO contributions for visualization)

print("\n" + "=" * 60)
print("Downloading L4 data...")
print("=" * 60)

zip_file_l4 = download_flights(
    BASE_URL, auth_headers,
    start=SEARCH_PARAMS["start"],
    end=SEARCH_PARAMS["end"],
    bbox=SEARCH_PARAMS["bbox"],
    level="L4",
    region=SEARCH_PARAMS["region"],
    l4_type=["CO_CONTRIBUTIONS"],
    output_dir=str(DATA_DIR)
)

print(f"\nL4 ZIP file: {zip_file_l4}")

### Analyze CO data across all flights

Now we use `rank_flights_by_co()` to:
1. Open each L2 file and check if **CO_P1** contains valid data
2. If L2 has no valid CO (variable missing, all NaN, or all -9999), fall back to the matching **L1 file**
3. Record the **max** and **mean** CO concentration for each flight
4. Sort all flights by **max CO descending** and display the **top 25**

This identifies the most interesting flights for further visualization.

In [ ]:
# Analyze all flights and rank by CO concentration
#
# This processes every L2 file in the ZIP, checks for valid CO_P1 data,
# and falls back to L1 when L2 has no CO data

print("=" * 60)
print("Analyzing CO data across all flights...")
print("=" * 60)

ranked_flights = rank_flights_by_co(zip_file_l2, zip_file_l1)

# Display the top 25 flights
top_n = 25
top_flights = ranked_flights[:top_n]

def fmt_avail(v):
    if v is None:
        return "-"
    return f"{v * 100:.0f}%"

print(f"\n{'=' * 210}")
print(f"  TOP {top_n} FLIGHTS BY MAX CO CONCENTRATION")
print(f"{'=' * 210}")
print(f"{'Rank':>4}  {'Flight ID':<20} {'Phase':<10} {'Airport':<25} {'Max':>8} {'Alt Max':>8} {'Mean':>8} {'Median':>8} {'P25':>8} {'P75':>8} {'Source':>6} {'CO Avail':>9}  {'Other Species':<30}")
print(f"{'':>4}  {'':20} {'':10} {'':25} {'(ppb)':>8} {'(m)':>8} {'(ppb)':>8} {'(ppb)':>8} {'(ppb)':>8} {'(ppb)':>8}")
print(f"{'-' * 210}")

for i, flight in enumerate(top_flights, 1):
    airport_short = flight['airport'][:25] if len(flight['airport']) > 25 else flight['airport']
    species = flight.get('other_species', '')
    co_avail_str = fmt_avail(flight.get('co_availability'))
    alt_max = flight.get('alt_max_co', 0.0)
    print(f"{i:>4}  {flight['flight_id']:<20} {flight['phase']:<10} {airport_short:<25} {flight['max_co']:>8.1f} {alt_max:>8.0f} {flight['mean_co']:>8.1f} {flight['median_co']:>8.1f} {flight['p25_co']:>8.1f} {flight['p75_co']:>8.1f} {flight['source']:>6} {co_avail_str:>9}  {species:<30}")

print(f"{'-' * 210}")
print("\nSource legend: L2 = validated data, L1 = raw data (L2 had no valid CO)")
print("CO Avail: percentage of valid CO_P1 data points (from NetCDF availability attribute)")
print(f"Total flights analyzed: {len(ranked_flights)}")

# Export all ranked flights to CSV
df = pd.DataFrame(ranked_flights)
csv_path = str(DATA_DIR / "ranked_flights_by_co.csv")
df.to_csv(csv_path, index=False)
print(f"\nCSV exported: {csv_path}")

## Data Visualization

Now we visualize the CO profiles for the top flights. The plots show CO_P1 vs altitude.

In [ ]:
# =============================================================================
# PLOT SELECTION - Choose which flights to plot
# =============================================================================
# Option 1: Plot the top 6 flights (default)
#   PLOT_SELECTION = None
#
# Option 2: Plot specific flights by their rank in the table above
PLOT_SELECTION = [1, 2, 3, 7, 12, 20, 25]

#PLOT_SELECTION = None  # Set to a list of ranks to pick specific flights

if PLOT_SELECTION is not None:
    plotted_flights = [ranked_flights[r - 1] for r in PLOT_SELECTION if r - 1 < len(ranked_flights)]
else:
    plotted_flights = ranked_flights[:6]

n_plots = len(plotted_flights)

# Plot CO profiles
# The grid follows the number of selected flights: a fixed 2x3 holds only six, and
# PLOT_SELECTION with more entries raised "index 6 is out of bounds".
n_cols = min(3, n_plots)
n_rows = math.ceil(n_plots / n_cols)
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(18, 5.0 * n_rows), squeeze=False)
axes = axes.flatten()

print(f"Plotting CO profiles for {n_plots} flights...\n")

for i, flight in enumerate(plotted_flights):
    ax = axes[i]
    flight_id = flight['flight_id']
    airport = flight['airport']
    source = flight['source']
    rank = PLOT_SELECTION[i] if PLOT_SELECTION else i + 1

    try:
        if source == "L2" and flight.get('l2_filename'):
            with zipfile.ZipFile(zip_file_l2, 'r') as z:
                with z.open(flight['l2_filename']) as f:
                    ds = xr.open_dataset(io.BytesIO(f.read()))
        else:
            with zipfile.ZipFile(zip_file_l1, 'r') as z:
                with z.open(flight['l1_filename']) as f:
                    ds = xr.open_dataset(io.BytesIO(f.read()))

        if 'CO_P1' in ds and 'baro_alt_AC' in ds:
            altitude = ds['baro_alt_AC'].values.astype(float)
            ax.plot(ds['CO_P1'], altitude, color='grey', linewidth=2, alpha=0.8)

        ds.close()

    except Exception as e:
        print(f"  Error plotting flight {flight_id}: {e}")

    airport_str = f" ({airport})" if airport and airport != 'N/A' else ""
    ax.set_title(f"#{rank} - {flight_id}{airport_str}\nMax CO: {flight['max_co']:.1f} ppb [{source}]", fontsize=9)
    ax.set_xlabel('CO (ppb)', fontsize=8)
    ax.set_ylabel('Altitude (m)', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='both', labelsize=7)

# Hide unused subplots
for j in range(n_plots, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Top Flights by Maximum CO Concentration (CO_P1)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
DRAWN_FIGURES["co_profiles"] = fig
plt.show()

print("\nCO profile plots complete!")
print("Red = L2 (validated), Pink = L1 (raw fallback)")

### Plot CO contributions by region for the top flights

For the flights with the highest CO concentration, we plot:
- The measured CO (CO_P1 from L2 or L1) as a reference line
- The L4 regional fire contributions (CO_GFAS_*) showing which GFED regions contributed most

This helps identify the geographic origin of the CO detected during the flight.

In [ ]:
# Plot CO contributions by region for the selected flights
# (uses plotted_flights from the cell above)
#
# X axis: CO (ppb), Y axis: Altitude (m)
# All 14 GFED regions are checked; only those with max > 0.1 ppb are plotted

n_plots = len(plotted_flights)

# The grid follows the number of selected flights: a fixed 2x3 holds only six, and
# PLOT_SELECTION with more entries raised "index 6 is out of bounds".
n_cols = min(3, n_plots)
n_rows = math.ceil(n_plots / n_cols)
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(20, 7.0 * n_rows), squeeze=False)
axes = axes.flatten()

all_regions = {
    'AUST': 'Australia',
    'BOAS': 'Boreal Asia',
    'BONA': 'Boreal N. America',
    'CEAM': 'Central America',
    'CEAS': 'Central Asia',
    'EQAS': 'Equatorial Asia',
    'EURO': 'Europe',
    'MIDE': 'Middle East',
    'NHAF': 'N. Hemisphere Africa',
    'NHSA': 'N. Hemisphere S. America',
    'SEAS': 'Southeast Asia',
    'SHAF': 'S. Hemisphere Africa',
    'SHSA': 'S. Hemisphere S. America',
    'TENA': 'Temperate N. America'
}

COLOR_HEX_BY_GFED4_REGION = {
    'BONA': '#3460ff',
    'TENA': '#ffab00',
    'CEAM': '#fe01fd',
    'NHSA': '#d8b6ff',
    'SHSA': '#956aff',
    'EURO': '#54ffff',
    'MIDE': '#488db1',
    'NHAF': '#547818',
    'SHAF': '#ff547e',
    'BOAS': '#ffff7e',
    'CEAS': '#aa8449',
    'SEAS': '#00d800',
    'EQAS': '#00acff',
    'AUST': '#e5ffbb',
}

print(f"Plotting regional CO contributions for {n_plots} flights...\n")

for i, flight in enumerate(plotted_flights):
    ax = axes[i]
    flight_id = flight['flight_id']
    phase = flight['phase']
    source = flight['source']
    airport = flight['airport']
    rank = PLOT_SELECTION[i] if PLOT_SELECTION else i + 1

    print(f"--- Flight #{rank}: {flight_id} ({phase}) | Source: {source} | Max CO: {flight['max_co']:.1f} ppb ---")

    ds_co = None
    ds_l4 = None

    try:
        # Load measured CO data (from L2 or L1) - in memory
        if source == "L2" and flight.get('l2_filename'):
            with zipfile.ZipFile(zip_file_l2, 'r') as z:
                with z.open(flight['l2_filename']) as f:
                    ds_co = xr.open_dataset(io.BytesIO(f.read()))
        else:
            with zipfile.ZipFile(zip_file_l1, 'r') as z:
                with z.open(flight['l1_filename']) as f:
                    ds_co = xr.open_dataset(io.BytesIO(f.read()))

        # Load matching L4 file - in memory
        with zipfile.ZipFile(zip_file_l4, 'r') as z:
            matching_l4 = [f for f in z.namelist() if flight_id in f and (phase == 'N/A' or f.endswith(f"-{phase}.nc4"))]
            if matching_l4:
                with z.open(matching_l4[0]) as f:
                    ds_l4 = xr.open_dataset(io.BytesIO(f.read()))

        # Get altitude
        altitude = None
        if ds_co is not None and 'baro_alt_AC' in ds_co:
            altitude = ds_co['baro_alt_AC'].values.astype(float)

        # Plot measured CO
        if ds_co is not None and 'CO_P1' in ds_co and altitude is not None:
            co_measured = ds_co['CO_P1'].values.astype(float)
            ax.plot(co_measured, altitude,
                    color='grey', linewidth=2,
                    label=f'{source} CO', alpha=0.8)

        # Plot regional fire contributions from L4
        if ds_l4 is not None and altitude is not None:
            alt_l4 = ds_l4['baro_alt_AC'].values.astype(float) if 'baro_alt_AC' in ds_l4 else altitude

            for region, name in all_regions.items():
                var_name = f'CO_GFAS_{region}'
                if var_name in ds_l4:
                    co_contrib = ds_l4[var_name].values.astype(float)
                    valid = co_contrib[~np.isnan(co_contrib)]
                    if len(valid) > 0 and np.max(valid) > 0.1:
                        region_color = COLOR_HEX_BY_GFED4_REGION.get(region, 'gray')
                        ax.plot(co_contrib, alt_l4, linewidth=1.5, label=region, alpha=0.8, color=region_color)

    except Exception as e:
        print(f"  Error: {e}")
    finally:
        if ds_co is not None:
            ds_co.close()
        if ds_l4 is not None:
            ds_l4.close()

    airport_short = airport[:25] if len(airport) > 25 else airport
    airport_str = f" ({airport_short})" if airport and airport != 'N/A' else ""
    phase_str = f" ({phase})" if phase and phase != 'N/A' else ""
    ax.set_title(f"#{rank} - {flight_id}{phase_str}{airport_str}\n"
                 f"Max CO: {flight['max_co']:.1f} ppb [{source}]", fontsize=9)
    ax.set_xlabel('CO (ppb)', fontsize=8)
    ax.set_ylabel('Altitude (m)', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right', fontsize=12)
    ax.tick_params(axis='both', labelsize=7)

# Hide unused subplots
for j in range(n_plots, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Top Flights - CO Fire Contributions by Region (GFAS)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
DRAWN_FIGURES["co_regional_contributions"] = fig
plt.show()

print("\nRegional contribution plots complete!")
print("Grey line: measured CO | Colored lines: L4 regional fire contributions (GFAS)")

<a id="export"></a>
## Take the results with you

The downloaded archives, the ranked-flights table and the figures are all in the `data/` folder
next to this notebook, **which is emptied every time the notebook is run again** — as in the other
tools of this VRE.

The cell below packs what you want to keep into a single zip in `exports/` and gives you a download
link. Figures that were only displayed inline are written out at that point, so there is no need to
decide in advance whether to keep them.

* **Figures and tables only** — the PNG and CSV files. Small.
* **Everything** — the downloaded L1/L2/L4 ZIP archives as well, which can run to hundreds of
  megabytes.

`exports/` is emptied at the start of each run too, so download what you need before starting a new
search.

In [ ]:
# Everything this notebook downloaded or produced lives in DATA_DIR. JupyterLab can only
# download one file at a time, so this packs it into a single zip and gives you a link.

import os
import zipfile

from IPython.display import HTML

RESULT_SUFFIXES = (".png", ".csv", ".txt")


def _server_root():
    """Directory the Jupyter server serves files from, which /files/ URLs are relative to.

    Assuming it is the home directory is usually right but not always. The running server
    records it in jpserver-*.json in the kernel's runtime directory, so that is asked first;
    walking up for a project marker is only a fallback, and deliberately does not override it
    (markers such as environment.yml also exist inside the tool folders, which would produce a
    root far below the real one and a broken link).
    """
    cwd = Path.cwd()
    try:
        import json as _json
        from ipykernel.connect import get_connection_file
        runtime_dir = Path(get_connection_file()).parent
        for server_file in runtime_dir.glob("jpserver-*.json"):
            try:
                root = Path(_json.loads(server_file.read_text(encoding="utf-8"))["root_dir"])
            except Exception:  # noqa: BLE001 - a malformed file is simply not a candidate
                continue
            if len(root.parts) > 1 and cwd.is_relative_to(root):
                return root
    except Exception:  # noqa: BLE001 - not running under ipykernel, or no runtime dir
        pass

    for marker in (".git", "environment.yml", "pyproject.toml", "setup.py"):
        current = cwd
        while current != current.parent:
            if (current / marker).exists() and len(current.parts) > 1:
                return current
            current = current.parent
    return Path.home()


def _download_link(archive):
    """A URL the browser will download, rather than JupyterLab try to open."""
    root = _server_root()
    try:
        rel = archive.resolve().relative_to(root)
    except ValueError:
        rel = Path(archive.name)
    # Under JupyterHub every URL carries a /user/<name>/ prefix; without it the link reaches
    # the hub instead of this server and comes back as an HTML error page.
    prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/").rstrip("/")
    return f"{prefix}/files/{rel.as_posix()}?download=1"


def _human(n):
    for unit in ("B", "kB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024


bundle_dd = widgets.Dropdown(
    options=[("Figures and tables only (PNG, CSV)", "results"),
             ("Everything, including the downloaded ZIP archives", "all")],
    value="results", description="Include:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="500px"))
bundle_btn = widgets.Button(description="Create archive", button_style="success",
                            icon="file-archive-o")
bundle_out = widgets.Output()


def _make_bundle(_=None):
    with bundle_out:
        bundle_out.clear_output()

        # Write out the figures drawn in this session; they are only shown inline otherwise
        written = []
        for stem, fig in DRAWN_FIGURES.items():
            path = DATA_DIR / f"{stem}.png"
            if not path.exists():
                fig.savefig(path, dpi=150, bbox_inches="tight")
                written.append(path.name)
        if written:
            print(f"Wrote {len(written)} figure(s): {', '.join(written)}")

        results_only = bundle_dd.value == "results"
        suffixes = RESULT_SUFFIXES if results_only else None
        files = [f for f in sorted(DATA_DIR.rglob("*"))
                 if f.is_file() and (suffixes is None or f.suffix.lower() in suffixes)]
        if not files:
            print("Nothing to pack yet: no figure has been drawn and no file has been "
                  "downloaded in this session.")
            return

        archive = EXPORT_DIR / f"iagos_results{'' if results_only else '_full'}.zip"
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
            for f in files:
                zf.write(f, f.relative_to(DATA_DIR))

        total = sum(f.stat().st_size for f in files)
        print(f"{len(files)} file(s), {_human(total)} -> {_human(archive.stat().st_size)} compressed")
        for f in files[:10]:
            print(f"   {f.relative_to(DATA_DIR)}")
        if len(files) > 10:
            print(f"   ... and {len(files) - 10} more")

        link = _download_link(archive)
        display(HTML(
            f'<p style="margin-top:8px"><a href="{link}" download '
            f'target="_blank" style="padding:8px 14px;background:#1B4053;color:white;'
            f'text-decoration:none;border-radius:5px;display:inline-block">'
            f'Download {archive.name}</a></p>'
            f'<p style="font-size:90%;color:#555">If the link does nothing, right-click '
            f'<code>{archive.name}</code> in the file browser on the left and choose '
            f'<i>Download</i>.</p>'))


bundle_dd.observe(lambda _c: bundle_out.clear_output(), names="value")
bundle_btn.on_click(_make_bundle)
display(widgets.VBox([bundle_dd, bundle_btn, bundle_out]))
print("Choose what to include, then press 'Create archive'.")